[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S13/2_Practica/en_clase/S13_01_visualizacion_resultados_en_clase.ipynb)

# S13 · 01 — Contar lo que hace el modelo: visualización de resultados · versión en clase

⏱ 35 min en clase · Módulo M5 (Visualización de resultados)

**Contexto TurisData.** Ya tenemos un modelo que predice qué reservas se cancelarán. Pero dirección no quiere oír *"AUC = 0,9"*: quiere saber **si puede fiarse, qué mueve las cancelaciones y qué decisión tomar el lunes**. En este laboratorio generamos las visualizaciones estándar de un modelo de clasificación y aprendemos a convertirlas en un mensaje para personas no técnicas.

**Este cuaderno tiene 4 ejercicios** (1 a 4), los que se hacen en el aula. El ejercicio 5 (el gráfico para dirección), las buenas prácticas de comunicación, el mini-reto del panel de una página y las preguntas de reflexión están en el cuaderno completo. Sus ejercicios 1, 2, 3 y 4 son los 4 de este.

**Cómo usar este cuaderno** (tiene dos partes):

- **Parte A · Ejecutar** (aquí abajo): de arriba abajo, cada celda de código lleva encima su título con un símbolo:
    - ⚙️ **Preparación**: se ejecuta sola, pulsa ▶ y sigue.
    - 👀 **Ejemplo**: pulsa ▶, mira el resultado y responde las preguntas 🧠 (no hay que escribir código). Las respuestas están en la Parte B.
    - ✏️ **Ejercicio**: tu turno. Cambia los `...` y pulsa ▶. Si sale **✔**, responde las preguntas 🧠 y sigues. Para comprobar tu código, tienes la solución de referencia ✅ en la Parte B.
- **Parte B · Explicación** (al final): para cada celda, **qué hace**, **qué resultado debe salir** y, en los ejemplos, las **respuestas** a las preguntas 🧠 y, en los ejercicios, la **solución de referencia** ✅ para comprobar tu código. Ábrela cuando quieras entender una celda, comprobar tus respuestas o no te salga. Usa el índice de la izquierda (☰) para saltar.

# Parte A · Ejecutar

### ⚙️ A1 · PREPARACIÓN — se ejecuta sola

Pulsa ▶ y sigue. No hace falta leerla.

In [ ]:
#@title A1 · ▶ Ejecuta esta celda primero (no hace falta leerla)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, roc_curve, roc_auc_score,
                             precision_recall_curve, average_precision_score, recall_score)
from sklearn.inspection import permutation_importance

SEMILLA = 42
np.random.seed(SEMILLA)
sns.set_theme(style="whitegrid", font_scale=1.0)

def euros(x):
    """4980 → '4.980 €' (puntos para los miles, como se escribe en español)."""
    return f"{x:,.0f}".replace(",", ".") + " €"
print("Listo ✔")

### 👀 A2 · EJEMPLO — Entrenar la regresión logística y el bosque aleatorio

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A2).

1. ¿Cuántas reservas se usan para probar y qué proporción se cancela?
2. ¿Qué guardan `prob_lr` y `prob_rf`?
3. ¿Qué dos modelos se comparan?

In [ ]:
reservas = pd.read_csv(dato("reservas_turisdata.csv"), parse_dates=["fecha_reserva", "fecha_llegada"])
reservas["mes_llegada"] = reservas["fecha_llegada"].dt.month
X = reservas.drop(columns=["id_reserva", "fecha_reserva", "fecha_llegada", "cancelada"])
y = reservas["cancelada"]
X_ent, X_test, y_ent, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEMILLA)

categoricas = X.select_dtypes("object").columns.tolist()
numericas = [c for c in X.columns if c not in categoricas]
prep = ColumnTransformer([("num", StandardScaler(), numericas),
                          ("cat", OneHotEncoder(handle_unknown="ignore"), categoricas)])

modelo_lr = Pipeline([("prep", prep), ("m", LogisticRegression(max_iter=2000))]).fit(X_ent, y_ent)
modelo_rf = Pipeline([("prep", prep), ("m", RandomForestClassifier(300, min_samples_leaf=5, random_state=SEMILLA, n_jobs=2))]).fit(X_ent, y_ent)

prob_lr = modelo_lr.predict_proba(X_test)[:, 1]     # probabilidad de cancelar
prob_rf = modelo_rf.predict_proba(X_test)[:, 1]
print(f"Reservas de prueba: {len(y_test)} · tasa real de cancelación: {y_test.mean():.1%}")

### ✏️ A3 · EJERCICIO 1 de 4 — Matriz de confusión normalizada y recall

**Tu turno:** Calcula `cm` con `confusion_matrix(y_test, pred_lr, normalize="true")`, siendo `pred_lr` la predicción de la logística con umbral 0,5 (`prob_lr >= 0.5`). Guarda en `recall_cancelaciones` la proporción de cancelaciones reales que el modelo detecta (fila de las cancelaciones, columna "cancela").

**Debe salir:** `el modelo detecta el 62% de las cancelaciones` y ✔ Ejercicio 1 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A3).

1. ¿Qué porcentaje de las cancelaciones detecta el modelo con umbral 0,5?
2. ¿Qué es un falso negativo en este problema?
3. ¿Por qué se normaliza por filas?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A3, **después de intentarlo**.

In [ ]:
# pista: convierte la comparación prob_lr >= 0.5 en enteros; el recall de la clase 1 es un elemento de la matriz normalizada por filas
pred_lr = ...
cm = ...
recall_cancelaciones = ...

assert cm.shape == (2, 2) and np.allclose(cm.sum(axis=1), 1), "cm debe ser 2x2 y cada fila sumar 1 (normalize='true')"
assert abs(recall_cancelaciones - recall_score(y_test, pred_lr)) < 1e-9, "recall_cancelaciones = fila 1, columna 1 de cm"
print(f"el modelo detecta el {recall_cancelaciones:.0%} de las cancelaciones")
print("✔ Ejercicio 1 correcto")

### 👀 A4 · EJEMPLO — Dibujar la matriz de confusión

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A4).

1. ¿Qué porcentaje de las reservas que no se cancelan acierta el modelo?
2. ¿Qué casilla muestra las cancelaciones que se escapan?
3. ¿Qué indica la diagonal de la matriz?

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 4))
ConfusionMatrixDisplay(cm, display_labels=["No cancela", "Cancela"]).plot(ax=ax, cmap="Blues", values_format=".0%", colorbar=False)
ax.set_title("Matriz de confusión (% de cada clase real)")
ax.set_xlabel("Lo que dijo el modelo"); ax.set_ylabel("Lo que ocurrió")
plt.tight_layout(); plt.show()

### ✏️ A5 · EJERCICIO 2 de 4 — AUC y precisión media de los dos modelos

**Tu turno:** Calcula `auc_lr` y `auc_rf` (`roc_auc_score`) y `ap_lr` y `ap_rf` (`average_precision_score`, el área de la curva PR) para ambos modelos con las probabilidades `prob_lr` y `prob_rf`. Guarda en `mejor_modelo` el texto `"Regresión logística"` o `"Bosque aleatorio"` según cuál tenga mayor AUC.

**Debe salir:** `AUC logística 0.744 · bosque 0.733 → Regresión logística` y ✔ Ejercicio 2 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A5).

1. ¿Qué AUC tiene cada modelo?
2. ¿Qué significa un AUC de 0,5?
3. ¿Declararías un ganador claro?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A5, **después de intentarlo**.

In [ ]:
# pista: ambas funciones reciben (valores reales, probabilidades); compara los dos AUC con un if/else
auc_lr = ...
auc_rf = ...
ap_lr = ...
ap_rf = ...
mejor_modelo = ...

assert 0.6 < auc_lr < 0.9 and 0.6 < auc_rf < 0.9, "Los AUC deberían estar entre 0,6 y 0,9; usa y_test y las probabilidades"
assert mejor_modelo in ("Regresión logística", "Bosque aleatorio"), "mejor_modelo debe ser uno de los dos textos indicados"
print(f"AUC logística {auc_lr:.3f} · bosque {auc_rf:.3f} → {mejor_modelo}")
print("✔ Ejercicio 2 correcto")

### 👀 A6 · EJEMPLO — Curvas ROC y PR de los dos modelos en un gráfico

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A6).

1. ¿Qué se ve cuando dos curvas casi se superponen?
2. ¿Qué representa la diagonal de la curva ROC?
3. ¿Qué representa la línea horizontal de la curva PR?

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
for nombre, prob, color in (("Regresión logística", prob_lr, "tab:blue"), ("Bosque aleatorio", prob_rf, "tab:orange")):
    fpr, tpr, _ = roc_curve(y_test, prob)
    ax[0].plot(fpr, tpr, color=color, label=f"{nombre} (AUC = {roc_auc_score(y_test, prob):.2f})")
    prec, rec, _ = precision_recall_curve(y_test, prob)
    ax[1].plot(rec, prec, color=color, label=f"{nombre} (AP = {average_precision_score(y_test, prob):.2f})")
ax[0].plot([0, 1], [0, 1], "--", color="gray", label="Azar")
ax[0].set_xlabel("Falsas alarmas (% de no cancelaciones avisadas)"); ax[0].set_ylabel("Cancelaciones detectadas")
ax[0].set_title("Curva ROC"); ax[0].legend(loc="lower right")
ax[1].axhline(y_test.mean(), linestyle="--", color="gray", label="Azar (tasa de cancelación)")
ax[1].set_xlabel("Exhaustividad (cancelaciones detectadas)"); ax[1].set_ylabel("Precisión (avisos que eran ciertos)")
ax[1].set_title("Curva precisión-exhaustividad"); ax[1].legend(loc="upper right"); ax[1].set_ylim(0, 1)
plt.tight_layout(); plt.show()

### ✏️ A7 · EJERCICIO 3 de 4 — Importancia por permutación

**Tu turno:** Calcula la importancia por permutación de `modelo_lr` sobre `X_test`/`y_test` con `permutation_importance(..., scoring="roc_auc", n_repeats=5, random_state=SEMILLA)`. Guarda en la `Series` `importancias` la media (`importances_mean`) con el nombre de cada variable como índice, ordenada de mayor a menor, y en `top3` la lista de las 3 primeras variables.

**Debe salir:** `las 3 más importantes: ['antelacion_dias', 'deposito', 'canal']` y ✔ Ejercicio 3 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A7).

1. ¿Cuáles son las tres variables más importantes?
2. ¿Cuánto cae el AUC al desordenar la antelación?
3. ¿Por qué se calcula sobre las variables originales y no sobre las columnas transformadas?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A7, **después de intentarlo**.

In [ ]:
# pista: el resultado tiene .importances_mean; conviértelo en pd.Series con index=X_test.columns y ordénalo
res = ...
importancias = ...
top3 = ...

assert isinstance(importancias, pd.Series) and len(importancias) == X_test.shape[1], "importancias: una Series con una entrada por variable"
assert importancias.is_monotonic_decreasing, "Ordena de mayor a menor"
assert len(top3) == 3 and set(top3) <= set(X_test.columns) and top3[0] == importancias.index[0], "top3: los nombres de las 3 primeras variables de importancias"
assert importancias.iloc[0] > 0.02, "La variable más importante debería hacer caer el AUC de forma apreciable; revisa el cálculo (¿desordenas la columna de X_test y mides con y_test?)"
print("las 3 más importantes:", top3)
print("✔ Ejercicio 3 correcto")

### 👀 A8 · EJEMPLO — Dibujar las variables que más pesan

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A8).

1. ¿Qué variable tiene la barra más larga?
2. ¿Qué ventaja tiene que el título diga la conclusión?

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
importancias.head(8)[::-1].plot.barh(ax=ax, color="tab:blue")
ax.set_xlabel("Caída del AUC al desordenar la variable")
ax.set_title(f"Lo que más pesa en una cancelación: {top3[0]}")
plt.tight_layout(); plt.show()

### ✏️ A9 · EJERCICIO 4 de 4 — El umbral que minimiza el coste

**Tu turno:** Para cada umbral de `umbrales` (de 0,05 a 0,95), calcula el coste total con las probabilidades `prob_lr`: `50 × falsos negativos + 30 × falsos positivos`. Guarda los costes en la lista `costes`, el mejor umbral en `umbral_optimo` y su coste en `coste_minimo`. Guarda también `coste_05`, el coste del umbral 0,5.

**Debe salir:** `umbral óptimo 0.35: ahorra 1.900 € frente al umbral 0,5 en esta muestra` y ✔ Ejercicio 4 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A9).

1. ¿Qué umbral minimiza el coste y cuánto se ahorra frente a 0,5?
2. ¿Por qué el umbral óptimo es menor que 0,5?
3. ¿Qué cuidado hay que tener con esa cifra de ahorro?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A9, **después de intentarlo**.

In [ ]:
umbrales = np.round(np.arange(0.05, 0.96, 0.01), 2)

def coste_total(prob, umbral, c_fn=50, c_fp=30):
    pred = (prob >= umbral).astype(int)
    fn = int(((pred == 0) & (y_test == 1)).sum())
    fp = int(((pred == 1) & (y_test == 0)).sum())
    return c_fn * fn + c_fp * fp

# pista: una lista por comprensión con la función coste_total ya escrita; np.argmin te da la posición del mínimo
costes = ...
umbral_optimo = ...
coste_minimo = ...
coste_05 = ...

assert len(costes) == len(umbrales), "Un coste por umbral"
assert 0.05 <= umbral_optimo <= 0.95 and coste_minimo == min(costes), "umbral_optimo debe ser uno de los umbrales y coste_minimo, el menor de los costes"
assert coste_minimo <= coste_05, "El coste mínimo no puede ser mayor que el del umbral 0,5"
print(f"umbral óptimo {umbral_optimo:.2f}: ahorra {euros(coste_05 - coste_minimo)} frente al umbral 0,5 en esta muestra")
print("✔ Ejercicio 4 correcto")

### 👀 A10 · EJEMPLO — Curva del coste según el umbral

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A10).

1. ¿Qué forma tiene la curva del coste?
2. ¿Qué coste tiene el umbral óptimo?
3. ¿Por qué sube el coste con umbrales muy altos?

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(umbrales, costes, color="tab:blue")
ax.plot(umbral_optimo, coste_minimo, "o", color="tab:red")
ax.annotate(f"Umbral óptimo {umbral_optimo:.2f}\n{euros(coste_minimo)}", (umbral_optimo, coste_minimo),
            xytext=(umbral_optimo + 0.12, coste_minimo + 0.08 * (max(costes) - min(costes))),
            arrowprops=dict(arrowstyle="->"))
ax.set_xlabel("Umbral (avisamos si la probabilidad supera este valor)"); ax.set_ylabel("Coste total en la muestra de prueba (€)")
ax.set_title(f"Coste total según el umbral: mínimo en {umbral_optimo:.2f}")
plt.tight_layout(); plt.show()

---

## 🏁 Fin del cuaderno de clase

El **[cuaderno completo](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S13/2_Practica/para_casa/S13_01_visualizacion_resultados.ipynb)** trae el ejercicio 5 (el gráfico para dirección), las buenas prácticas de comunicación, el mini-reto del panel de una página y las preguntas de reflexión, cada ejercicio con su solución plegada en la Parte B.

# Parte B · Explicación de cada celda y de cada resultado

Cada apartado lleva el **mismo número** que su celda de la Parte A (A1, A2…). Si una celda no te sale, mira su «Si falla».

#### A1 · Preparación — Preparar las herramientas

**Contexto.** Ya tenemos un modelo que predice qué reservas se cancelarán. Pero dirección no quiere oír «AUC = 0,9»: quiere saber **si puede fiarse, qué mueve las cancelaciones y qué decisión tomar el lunes**. Aquí generamos las visualizaciones estándar de un modelo de clasificación y las convertimos en un mensaje para personas no técnicas.

- **Qué hace:**
    - Carga `numpy`, `pandas`, `matplotlib`, `seaborn` y, de `scikit-learn`, el preprocesado, la regresión logística, el bosque aleatorio y las métricas (matriz de confusión, ROC, PR, `permutation_importance`).
    - Fija la semilla (`SEMILLA`), el estilo de los gráficos y define `euros(x)`, que escribe una cantidad con puntos de millar y el símbolo € (`4980` → `4.980 €`).
- **Qué debe salir:**
    - `Listo ✔`.
- **Cómo leerlo:**
    - No hay que entender esta celda: solo ejecutarla **la primera**, y otra vez si reinicias el entorno.
- **Si falla:**
    - `NameError: name 'np' is not defined` → no has ejecutado esta celda. Ejecútala y repite la que falló.

## 0. Punto de partida: dos modelos sencillos

#### A2 · Ejemplo — Entrenar la regresión logística y el bosque aleatorio

Predecimos `cancelada` (1 = la reserva se cancela). Entrenamos una **regresión logística** y un
**bosque aleatorio** con el 75 % de las reservas y guardamos el 25 % para evaluar.

- **Qué hace:**
    - Carga las reservas, añade `mes_llegada` y separa las variables (`X`) de lo que se predice (`y`, la columna `cancelada`); `train_test_split` guarda el 25 % para probar.
    - `prep` (un `ColumnTransformer`) escala las variables numéricas y convierte las categóricas en columnas 0/1.
    - `modelo_lr` (regresión logística) y `modelo_rf` (bosque aleatorio de 300 árboles) se entrenan con el 75 %; `prob_lr` y `prob_rf` son sus **probabilidades de cancelar** en la prueba.
- **Qué debe salir:**
    - `Reservas de prueba: 1500 · tasa real de cancelación: 45.9%`.
- **Cómo leerlo:**
    - Se prueban 1 500 reservas y cerca de la mitad (45,9 %) se cancelan. Todo lo que sigue usa `prob_lr` y `prob_rf`.
- **Si falla:**
    - `NameError: name 'dato' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántas reservas se usan para probar y qué proporción se cancela?* → **1 500** reservas y el **45,9 %** se cancelan.
2. *¿Qué guardan `prob_lr` y `prob_rf`?* → La **probabilidad de cancelar** de cada reserva de prueba, según cada modelo.
3. *¿Qué dos modelos se comparan?* → Una **regresión logística** y un **bosque aleatorio**.

</details>

## 1. Matriz de confusión

#### A3 · Ejercicio 1 — Matriz de confusión normalizada y recall

**Idea.** Cruza lo que **pasó** con lo que el modelo **dijo** (con un umbral, p. ej. "cancela si
probabilidad ≥ 0,5"): aciertos y dos tipos de error.
- **Falso negativo:** el modelo dijo "no cancela" y canceló → habitación vacía que se pudo revender.
- **Falso positivo:** dijo "cancela" y no canceló → molestamos al cliente o sobrevendemos sin motivo.

Para dirección, **normalizar por filas** (porcentajes de cada clase real) se lee mucho mejor que los
números brutos.

- **Qué hace:**
    - `(prob_lr >= 0.5).astype(int)` convierte la probabilidad en una decisión: 1 («cancela») si es al menos 0,5, y 0 si no (`pred_lr`).
    - `confusion_matrix(y_test, pred_lr, normalize="true")` cruza lo que pasó con lo que dijo el modelo, normalizado por filas: cada fila suma 1 (`cm`).
    - `cm[1, 1]` es la fila de las cancelaciones reales, columna «cancela»: la proporción detectada, el *recall* (`recall_cancelaciones`).
- **Qué debe salir:**
    - `el modelo detecta el 62% de las cancelaciones` y `✔ Ejercicio 1 correcto`.
- **Cómo leerlo:**
    - Con umbral 0,5 el modelo avisa del **62 %** de las cancelaciones y deja escapar el 38 % (falsos negativos). Para dirección, esos porcentajes se leen mejor que los números brutos.
- **Si falla:**
    - `cm debe ser 2x2 y cada fila sumar 1 (normalize='true')` → falta `normalize="true"` en `confusion_matrix`.
    - `recall_cancelaciones = fila 1, columna 1 de cm` → has cogido otro elemento de la matriz (el de las no cancelaciones, `cm[0, 0]`, u otra fila).
    - `NameError: name 'prob_lr' is not defined` → falta ejecutar el ejemplo de los dos modelos.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué porcentaje de las cancelaciones detecta el modelo con umbral 0,5?* → El **62 %**.
2. *¿Qué es un falso negativo en este problema?* → Una reserva que el modelo dijo que **no** se cancelaba y se canceló: habitación vacía que se pudo revender.
3. *¿Por qué se normaliza por filas?* → Para ver el **porcentaje de cada clase real** que se acierta o se falla, que se lee mejor que los números brutos.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
pred_lr = (prob_lr >= 0.5).astype(int)
cm = confusion_matrix(y_test, pred_lr, normalize="true")
recall_cancelaciones = cm[1, 1]
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

#### A4 · Ejemplo — Dibujar la matriz de confusión

- **Qué hace:**
    - `ConfusionMatrixDisplay(cm, display_labels=[...]).plot(...)` dibuja la matriz como una cuadrícula 2 × 2 con porcentajes (`values_format=".0%"`), en azules.
    - Los ejes dicen «Lo que ocurrió» (filas) y «Lo que dijo el modelo» (columnas).
- **Qué debe salir:**
    - Un gráfico «Matriz de confusión (% de cada clase real)» con cuatro casillas: `73 %` y `27 %` en la fila «No cancela»; `38 %` y `62 %` en la fila «Cancela».
- **Cómo leerlo:**
    - La diagonal son los aciertos (73 % y 62 %). Fuera de la diagonal están los errores: el 27 % de falsas alarmas y el 38 % de cancelaciones que no se vieron venir.
- **Si falla:**
    - `NameError: name 'cm' is not defined` → falta el ejercicio 1.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué porcentaje de las reservas que no se cancelan acierta el modelo?* → El **73 %**.
2. *¿Qué casilla muestra las cancelaciones que se escapan?* → La de la fila **«Cancela»** y columna **«No cancela»**: el **38 %**.
3. *¿Qué indica la diagonal de la matriz?* → Los **aciertos** de cada clase.

</details>

## 2. Curvas ROC y precisión-exhaustividad (PR)

#### A5 · Ejercicio 2 — AUC y precisión media de los dos modelos

**Idea.** La matriz depende de **un** umbral. Las curvas resumen **todos** los umbrales posibles:
- **ROC:** eje X = falsas alarmas (de los que no cancelan, ¿a cuántos avisamos?), eje Y = cancelaciones
  detectadas. La diagonal es el azar; cuanto más arriba-izquierda, mejor. **AUC** = área bajo la curva
  (0,5 = azar, 1 = perfecto).
- **PR:** eje X = exhaustividad (*recall*: ¿qué parte de las cancelaciones cazamos?), eje Y =
  precisión (de los avisos, ¿cuántos eran ciertos?). Es más informativa si la clase que buscamos es rara.

- **Qué hace:**
    - `roc_auc_score(y_test, prob)` da el área bajo la curva ROC de cada modelo (`auc_lr`, `auc_rf`): 0,5 es el azar y 1 es perfecto.
    - `average_precision_score(y_test, prob)` da el área bajo la curva PR (`ap_lr`, `ap_rf`).
    - `mejor_modelo` es el texto del modelo con mayor AUC (un `if/else` en una línea).
- **Qué debe salir:**
    - `AUC logística 0.744 · bosque 0.733 → Regresión logística` y `✔ Ejercicio 2 correcto`.
- **Cómo leerlo:**
    - Los dos modelos están muy cerca (0,744 y 0,733) y claramente por encima del azar (0,5), pero lejos de la perfección. Ni mucho menos «AUC = 0,9»: con estos datos, una diferencia de un punto no es un ganador claro. La precisión media (AP) es 0,72 y 0,70.
- **Si falla:**
    - `Los AUC deberían estar entre 0,6 y 0,9; usa y_test y las probabilidades` → has pasado las predicciones (0/1) en vez de las probabilidades, o los argumentos en orden cambiado.
    - `mejor_modelo debe ser uno de los dos textos indicados` → el texto no es exactamente `"Regresión logística"` o `"Bosque aleatorio"`.
    - `NameError: name 'prob_rf' is not defined` → falta ejecutar el ejemplo de los dos modelos.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué AUC tiene cada modelo?* → **0,744** la regresión logística y **0,733** el bosque aleatorio.
2. *¿Qué significa un AUC de 0,5?* → Que el modelo **no mejora el azar**.
3. *¿Declararías un ganador claro?* → **No**: la diferencia es de un punto, demasiado pequeña para decidir solo con ella.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
auc_lr, auc_rf = roc_auc_score(y_test, prob_lr), roc_auc_score(y_test, prob_rf)
ap_lr, ap_rf = average_precision_score(y_test, prob_lr), average_precision_score(y_test, prob_rf)
mejor_modelo = "Regresión logística" if auc_lr >= auc_rf else "Bosque aleatorio"
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

#### A6 · Ejemplo — Curvas ROC y PR de los dos modelos en un gráfico

- **Qué hace:**
    - Para cada modelo, `roc_curve` y `precision_recall_curve` calculan los puntos de las curvas con todos los umbrales; se dibujan con su AUC y su AP en la leyenda.
    - A la izquierda, la ROC con la diagonal del azar; a la derecha, la curva PR con una línea horizontal en la tasa de cancelación (el azar de PR).
- **Qué debe salir:**
    - Dos gráficos: «Curva ROC» (AUC 0,74 y 0,73) y «Curva precisión-exhaustividad» (AP 0,72 y 0,70), con dos curvas casi superpuestas en cada uno.
- **Cómo leerlo:**
    - Las curvas de los dos modelos casi se tocan: confirman que rinden parecido. Las dos quedan por encima del azar (la diagonal en ROC y la línea de 0,46 en PR).
- **Si falla:**
    - `NameError: name 'prob_lr' is not defined` → falta ejecutar el ejemplo de los dos modelos.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué se ve cuando dos curvas casi se superponen?* → Que los dos modelos rinden **casi igual**.
2. *¿Qué representa la diagonal de la curva ROC?* → El **azar**.
3. *¿Qué representa la línea horizontal de la curva PR?* → El **azar** de esa curva: la tasa de cancelación (≈ 0,46).

</details>

## 3. Importancia por permutación

#### A7 · Ejercicio 3 — Importancia por permutación

**Idea.** Se **desordena al azar una variable** en los datos de prueba y se mide **cuánto empeora** el
modelo. Si empeora mucho, esa variable era importante. Funciona con cualquier modelo y se calcula sobre
las **variables originales** (no sobre columnas transformadas), que es lo que dirección entiende.
(SHAP, que da explicaciones caso a caso, se verá en el proyecto; la idea es parecida.)

- **Qué hace:**
    - `permutation_importance(modelo_lr, X_test, y_test, scoring="roc_auc", n_repeats=5, random_state=SEMILLA)` desordena al azar cada variable de la prueba (5 veces) y mide cuánto cae el AUC.
    - `pd.Series(res.importances_mean, index=X_test.columns).sort_values(ascending=False)` pone la caída media de cada variable en una serie ordenada de mayor a menor (`importancias`).
    - `importancias.index[:3].tolist()` toma los nombres de las tres primeras (`top3`).
- **Qué debe salir:**
    - `las 3 más importantes: ['antelacion_dias', 'deposito', 'canal']` y `✔ Ejercicio 3 correcto`.
- **Cómo leerlo:**
    - Desordenar `antelacion_dias` hace caer el AUC en 0,10, mucho más que cualquier otra (`deposito` 0,045, `canal` 0,042, `tarifa` 0,031). Las demás pesan menos de 0,01.
- **Si falla:**
    - `importancias: una Series con una entrada por variable` → no has construido la `Series` con `index=X_test.columns` (o has usado las columnas transformadas).
    - `Ordena de mayor a menor` → falta `sort_values(ascending=False)`.
    - `top3: los nombres de las 3 primeras variables de importancias` → has guardado los valores en vez de los nombres (`importancias.index[:3]`).
    - `La variable más importante debería hacer caer el AUC de forma apreciable…` → no has desordenado con `X_test` y `y_test`.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuáles son las tres variables más importantes?* → **antelacion_dias**, **deposito** y **canal**.
2. *¿Cuánto cae el AUC al desordenar la antelación?* → Unas **0,10**.
3. *¿Por qué se calcula sobre las variables originales y no sobre las columnas transformadas?* → Porque es lo que **dirección entiende** («la antelación», no «la columna 7 tras el escalado»).

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
res = permutation_importance(modelo_lr, X_test, y_test, scoring="roc_auc", n_repeats=5, random_state=SEMILLA)
importancias = pd.Series(res.importances_mean, index=X_test.columns).sort_values(ascending=False)
top3 = importancias.index[:3].tolist()
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

#### A8 · Ejemplo — Dibujar las variables que más pesan

- **Qué hace:**
    - `importancias.head(8)[::-1].plot.barh(...)` dibuja las 8 variables más importantes en barras horizontales (invertidas para que la mayor quede arriba).
    - El título usa `top3[0]` para decir cuál es la que más pesa.
- **Qué debe salir:**
    - Un gráfico de barras horizontales «Lo que más pesa en una cancelación: antelacion_dias» con la barra de la antelación mucho más larga que el resto.
- **Cómo leerlo:**
    - Una barra larga y varias cortas: el mensaje está claro, y el título ya dice la conclusión.
- **Si falla:**
    - `NameError: name 'importancias' is not defined` → falta el ejercicio 3.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué variable tiene la barra más larga?* → **antelacion_dias**.
2. *¿Qué ventaja tiene que el título diga la conclusión?* → Que el gráfico **se entiende de un vistazo**, sin leer el resto.

</details>

## 4. Elegir el umbral con el coste del negocio

#### A9 · Ejercicio 4 — El umbral que minimiza el coste

**Idea.** El umbral 0,5 es una convención, no una decisión. Cada error cuesta distinto:
- **Falso negativo** (no lo vimos venir y canceló): habitación vacía que ya no se revende → **50 €** de ingreso perdido.
- **Falso positivo** (avisamos y no cancelaba): tratamos como "en riesgo" a un cliente que sí viene (compensación o descuento para que confirme) → **30 €**.

Recorremos todos los umbrales y elegimos el de **menor coste total**.

- **Qué hace:**
    - `umbrales` son los umbrales de 0,05 a 0,95; `coste_total(prob, umbral)` (ya escrita) cuenta los falsos negativos (50 € cada uno) y los falsos positivos (30 €) con ese umbral y devuelve el coste.
    - `[coste_total(prob_lr, u) for u in umbrales]` calcula el coste de cada umbral (`costes`).
    - `umbrales[int(np.argmin(costes))]` es el umbral de menor coste (`umbral_optimo`); `min(costes)` es su coste (`coste_minimo`) y `coste_total(prob_lr, 0.5)` el del umbral 0,5 (`coste_05`).
- **Qué debe salir:**
    - `umbral óptimo 0.35: ahorra 1.900 € frente al umbral 0,5 en esta muestra` y `✔ Ejercicio 4 correcto`.
- **Cómo leerlo:**
    - Con 0,5 el coste en la muestra es de 19.710 €; con 0,35 baja a 17.810 €: **1.900 € menos**. El umbral óptimo es más bajo que 0,5 porque no ver una cancelación (50 €) cuesta más que avisar de más (30 €). Es una cifra de esta muestra, no una promesa.
- **Si falla:**
    - `Un coste por umbral` → `costes` no tiene un valor por cada umbral de `umbrales`.
    - `umbral_optimo debe ser uno de los umbrales y coste_minimo, el menor de los costes` → has guardado la posición (`argmin`) en lugar del umbral, o has usado `max` en vez de `min`.
    - `El coste mínimo no puede ser mayor que el del umbral 0,5` → `coste_05` no usa `coste_total(prob_lr, 0.5)`.
    - `NameError: name 'y_test' is not defined` → falta ejecutar el ejemplo de los dos modelos.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué umbral minimiza el coste y cuánto se ahorra frente a 0,5?* → **0,35**, que ahorra **1.900 €** en esta muestra.
2. *¿Por qué el umbral óptimo es menor que 0,5?* → Porque un **falso negativo cuesta 50 €** y un falso positivo solo 30 €: conviene avisar más.
3. *¿Qué cuidado hay que tener con esa cifra de ahorro?* → Es de **esta muestra de prueba**: no es una promesa para el futuro.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
umbrales = np.round(np.arange(0.05, 0.96, 0.01), 2)

def coste_total(prob, umbral, c_fn=50, c_fp=30):
    pred = (prob >= umbral).astype(int)
    fn = int(((pred == 0) & (y_test == 1)).sum())
    fp = int(((pred == 1) & (y_test == 0)).sum())
    return c_fn * fn + c_fp * fp

costes = [coste_total(prob_lr, u) for u in umbrales]
umbral_optimo = umbrales[int(np.argmin(costes))]
coste_minimo = min(costes)
coste_05 = coste_total(prob_lr, 0.5)
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

#### A10 · Ejemplo — Curva del coste según el umbral

- **Qué hace:**
    - Dibuja el coste total para cada umbral, marca el mínimo con un punto rojo y lo anota con el umbral y su coste (con `euros`).
- **Qué debe salir:**
    - Un gráfico «Coste total según el umbral: mínimo en 0.35» con una curva en forma de U y una anotación «Umbral óptimo 0.35 / 17.810 €».
- **Cómo leerlo:**
    - Con umbrales muy bajos se avisa de casi todos (muchos falsos positivos) y con muy altos de casi ninguno (muchos falsos negativos): el coste sube por los dos lados y el mínimo queda en medio.
- **Si falla:**
    - `NameError: name 'costes' is not defined` → falta el ejercicio 4.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué forma tiene la curva del coste?* → De **U**: sube con umbrales muy bajos y muy altos.
2. *¿Qué coste tiene el umbral óptimo?* → **17.810 €**.
3. *¿Por qué sube el coste con umbrales muy altos?* → Porque casi **no se avisa de ninguna cancelación**: muchos falsos negativos a 50 € cada uno.

</details>